# 03 Classification and validation

This notebook performs supervised classification and validation of the preprocessed spectra.

The workflow includes:

- loading the preprocessed spectral dataset,
- defining classification tasks with different levels of specificity,
- dimensionality reduction using Principal Component Analysis (PCA),
- supervised classification using a Support Vector Classifier (SVC),
- Leave-One-Group-Out cross-validation (LOGO-CV),
- evaluation using confusion matrices and classification metrics,
- and diagnostic analysis of measurement-series-specific batch effects.

The classification tasks progressively reduce the level of material specificity, ranging from fine-grained sample differentiation to binary plastic versus non-plastic discrimination.

## Import dependencies

Import the required analysis, preprocessing, plotting, and machine-learning utilities used throughout the classification workflow.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
import pandas as pd


from src.cache import (
    load_spectral_dataset_cache,
)

from src.classification import (
    create_pca_svc_pipeline,
    evaluate_logo_predictions,
    get_logo_pca_component_counts,
    add_sample_metadata,
    make_class_labels,
    make_polymer_family_labels,
    make_binary_plastic_labels,
    export_prediction_results,
    evaluate_generalisable_batch_effects,
)

from src.plotting import (
    plot_confusion_matrix,
    plot_pca_component_distribution,
)

THRESHOLD_FACTOR = 1.05

pipeline = create_pca_svc_pipeline(
    pca_variance=0.99
)

## Define classification tasks

Three classification tasks are defined:

1. Full multiclass differentiation between all individual sample classes.
2. Reduced polymer-family-level differentiation with merged colour variants and related material subclasses.
3. Binary differentiation between plastic and non-plastic materials.

In the reduced classification task, colour variants are merged within each polymer family. PE-HD samples are additionally merged with PE due to their closely related polymer chemistry.

The display order for each classification task can be defined independently in the following section.

In [ ]:
display_order_all = [
    "Test tube",
    "Microscope slide",
    "Wood",
    "Sand",
    "PE",
    "PE-HD",
    "PE-HD (blue)",
    "PP",
    "PP (green)",
    "PP (orange)",
    "PP (pink)",
    "PP (yellow)",
    "PS",
    "PS (blue)",
    "PS (red)",
    "PS (yellow)",
    "PET",
    "PVC",
]

display_order_reduced = [
    "Test tube",
    "Microscope slide",
    "Wood",
    "Sand",
    "PE",
    "PP",
    "PS",
    "PET",
    "PVC",
]

display_order_binary = [
    "Non-plastic",
    "Plastic",
]

## Load preprocessed spectra

Load the cached preprocessed spectral dataset generated in the preprocessing workflow.

The preprocessing pipeline includes:

- sequence-specific background correction,
- detector artefact interpolation,
- Standard Normal Variate (SNV) transformation,
- Savitzky–Golay first-derivative filtering with integrated smoothing.

These transformed first-derivative spectra are used as input for all subsequent classification analyses.

In [ ]:
preprocessed = load_spectral_dataset_cache(
    name=f"continuous_spectra_preprocessed_{str(THRESHOLD_FACTOR).replace('.', 'p')}"
)

## Prepare classification labels and pipeline

The spectral intensities are extracted as feature matrix `X`.

Classification labels are generated for:

- the full multiclass task,
- the reduced polymer-family task,
- and the binary plastic versus non-plastic task.

A machine-learning pipeline combining PCA and an RBF-kernel Support Vector Classifier (SVC) is then initialised.

In [ ]:
X = preprocessed.intensities
metadata = add_sample_metadata(preprocessed.metadata)

y_classes = make_class_labels(metadata)
y_reduced = make_polymer_family_labels(metadata)
y_binary = make_binary_plastic_labels(metadata)

groups = metadata["sequence_id"].to_numpy()

## Analyse retained PCA dimensionality

The number of retained principal components is evaluated for each Leave-One-Group-Out fold.

PCA is fitted independently within each training fold, and the number of components required to retain the specified variance threshold `pca_variance` in the `pipeline` is recorded. This analysis assesses the stability of the dimensionality reduction across measurement series.

In [ ]:
pc_counts = get_logo_pca_component_counts (
    X=X,
    y=y_classes,
    groups=groups,
    pipeline=pipeline
)

In [ ]:
fig = plot_pca_component_distribution(pc_counts)
fig.show()

## Task 1: Full multiclass classification

Evaluate the most demanding classification scenario using Leave-One-Group-Out cross-validation.

This task differentiates between all individual sample classes, including colour variants and subclasses within the same polymer family.

LOGO-CV uses complete measurement sequences as independent groups to avoid information leakage between training and test data.

In [ ]:
results_all = evaluate_logo_predictions(
    X=X,
    y=y_classes,
    groups=groups,
    pipeline=pipeline,
    n_jobs=-1,
)

print(results_all["classification_report"])

results_all_df = export_prediction_results(
    metadata=metadata,
    y_true=y_classes,
    y_pred=results_all["y_pred"],
    output_path=(
        PROJECT_ROOT
        / "results"
        / "results_all_predictions.csv"
    ),
)

In [ ]:
results_all_df = pd.read_csv(PROJECT_ROOT / "results" / "results_all_predictions.csv")
y_pred = results_all_df["y_pred"].to_numpy()
y_true = results_all_df["y_true"].to_numpy()

fig = plot_confusion_matrix(
    y_true=y_true,
    y_pred=y_pred,
    labels=display_order_all,
    title="Full 18-class discrimination confusion matrix",
    height=800,
)

fig.show()

## Task 2: Reduced polymer-family classification

Evaluate the reduced classification task in which colour and source variants are merged within broader polymer families.

This task focuses on polymer-family-level spectral characteristics rather than fine-grained sample differentiation.

In [ ]:
results_reduced = evaluate_logo_predictions(
    X=X,
    y=y_reduced,
    groups=groups,
    pipeline=pipeline,
)

print(results_reduced["classification_report"])

results_reduced_df = export_prediction_results(
    metadata=metadata,
    y_true=y_reduced,
    y_pred=results_reduced["y_pred"],
    output_path=(
        PROJECT_ROOT
        / "results"
        / "results_reduced_predictions.csv"
    ),
)

In [ ]:
results_reduced_df = pd.read_csv(PROJECT_ROOT / "results" / "results_reduced_predictions.csv")
y_pred = results_reduced_df["y_pred"].to_numpy()
y_true = results_reduced_df["y_true"].to_numpy()

fig = plot_confusion_matrix(
    y_true=y_true,
    y_pred=y_pred,
    labels=display_order_reduced,
    title="Reduced polymer-type classification confusion matrix",
    height=800,
)

fig.show()

## Task 3: Binary plastic versus non-plastic classification

Evaluate the binary differentiation between plastic and non-plastic materials.

This task reflects a detection-oriented scenario in which the primary objective is the reliable identification of polymer-based materials against natural or technical background materials.

In [ ]:
results_binary = evaluate_logo_predictions(
    X=X,
    y=y_binary,
    groups=groups,
    pipeline=pipeline,
)

print(results_binary["classification_report"])

results_binary_df = export_prediction_results(
    metadata=metadata,
    y_true=y_binary,
    y_pred=results_binary["y_pred"],
    output_path=(
        PROJECT_ROOT
        / "results"
        / "results_binary_predictions.csv"
    ),
)

In [ ]:
results_binary_df = pd.read_csv(PROJECT_ROOT / "results" / "results_binary_predictions.csv")
y_pred = results_binary_df["y_pred"].to_numpy()
y_true = results_binary_df["y_true"].to_numpy()

fig = plot_confusion_matrix(
    y_true=y_true,
    y_pred=y_pred,
    labels=display_order_binary,
    title="Binary plastic versus non-plastic differentiation confusion matrix",
    height=800,
)

fig.show()

## Analyse measurement-series-specific batch effects

Evaluate whether measurement-series identity remains detectable within individual material classes. For each material class, a separate classifier is trained to predict the originating measurement sequence using repeated stratified cross-validation.

Prediction accuracies substantially above the random baseline indicate the presence of systematic measurement-series-dependent structure within the processed spectra. This analysis serves as a diagnostic for potential batch effects and complements the LOGO-CV evaluation strategy.

In [ ]:
generalisation_report = evaluate_generalisable_batch_effects(
    X,
    y_classes,
    groups,
)

print(generalisation_report.to_string(index=False))